# Building an ML Model from Scratch
## Hands-on workshop: Iris classification

Goal: predict Iris species from four flower measurements.

**Workflow:** load data → explore → split → train → predict → evaluate.

The main model uses scikit-learn for the end-to-end workflow. The companion `src/knn_from_scratch.py` implements the core KNN prediction logic with NumPy.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

X, y = iris.data, iris.target

print('X shape:', X.shape)
print('Features:', list(X.columns))
print('Classes:', list(iris.target_names))
X.head()

X shape: (150, 4)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## 1. Understand features and labels
- **Features (`X`)** are input measurements.
- **Labels (`y`)** are the answers the model learns to predict.
- **Classification** predicts a category.

In [2]:
df = X.copy()

df['species'] = y.map(dict(enumerate(iris.target_names)))

display(df.head())
print(df['species'].value_counts())
display(df.describe())

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


## 2. Split the data
Train on one subset and evaluate on unseen examples. `stratify=y` keeps class proportions similar across the split.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Training rows:', len(X_train), 'Test rows:', len(X_test))

Training rows: 120 Test rows: 30


## 3. Train a model
KNN predicts the majority class among nearby training examples. Scaling helps because KNN relies on distances. The pipeline fits the scaler only on training data.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
model.fit(X_train, y_train)
print('Model trained!')

Model trained!


## 4. Evaluate predictions
Accuracy is the fraction of correct predictions. A confusion matrix shows actual vs. predicted classes.

In [5]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Accuracy: 0.933
[[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30



## 5. Predict a new flower

In [6]:
sample = pd.DataFrame([{'sepal length (cm)': 5.1, 'sepal width (cm)': 3.5, 'petal length (cm)': 1.4, 'petal width (cm)': 0.2}])
prediction = model.predict(sample)[0]

print('Predicted species:', iris.target_names[prediction])

Predicted species: setosa


## Participant challenges
1. Change `n_neighbors=5` to `3` and compare metrics.
2. Change `test_size=0.2` to `0.3` and count test rows.
3. Explain why the scaler is fitted only on training data.
4. Run `python src/knn_from_scratch.py` and compare results.

## Takeaways
- Separate features from labels.
- Split data before training.
- Fit preprocessing on training data only.
- Evaluate on unseen test data.
- Interpret metrics rather than relying on a single score.